In [ ]:
!git clone -b handson https://github.com/linnabraham/galactic-rings.git

In [1]:
%cd galactic-rings

/home/linn/2024/dec/aiml-handson/galactic-rings


/data/linn/miniconda3/envs/aiml/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
%%capture
!pip install wandb
!pip install gdown

In [ ]:
%%capture
!pip install tensorflow

## Imports

In [19]:
import os
import json
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from alexnet_utils.params import parser, print_arguments
from alexnet_utils.alexnet import AlexNet
import wandb

In [3]:
print(tf.__version__)

2.18.0


In [4]:
print(wandb.__version__)

0.19.1


In [5]:
print(tf.config.list_physical_devices('GPU'))

[]


W0000 00:00:1736010158.814095   11748 gpu_device.cc:2344] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Create datasets

In [10]:
!gdown --fuzzy "https://drive.google.com/file/d/1RlPl3WD4JDz5N-kx5g00YveLtZ43vPw-/view?usp=drive_link"

Downloading...
From (original): https://drive.google.com/uc?id=1RlPl3WD4JDz5N-kx5g00YveLtZ43vPw-
From (redirected): https://drive.google.com/uc?id=1RlPl3WD4JDz5N-kx5g00YveLtZ43vPw-&confirm=t&uuid=8133badd-4b5e-429d-bf5f-08a8fe52c446
To: /home/linn/2024/dec/aiml-handson/galactic-rings/galaxies.tar.gz
100%|██████████████████████████████████████| 77.6M/77.6M [00:05<00:00, 15.1MB/s]


In [11]:
!echo "0f456e955b0b5312aec8d2dd6186218c  galaxies.tar.gz" | md5sum -c

galaxies.tar.gz: OK


In [12]:
%%capture
!tar xvzf galaxies.tar.gz -C data/

## Define argparse arguments

In [6]:
parser.add_argument('-images', '--images', required=True, help="path containing images of two classes")
parser.add_argument('-epochs', '--epochs', required=True, type=int, default=50, help="num epochs")
parser.add_argument('-model-path', '--model-path', default=None, help="Filepath to save model during training and to load model from when testing")
parser.add_argument('-val-dir', '--val-dir', default=None, help="path containing validation data")
parser.add_argument('-retrain', '--retrain', action="store_true", help="Whether to continue previous training")

_StoreTrueAction(option_strings=['-retrain', '--retrain'], dest='retrain', nargs=0, const=True, default=False, type=None, choices=None, required=False, help='Whether to continue previous training', metavar=None)

In [7]:
args = parser.parse_args(['-images', 'data/galaxies', '-epochs', '2'])

In [8]:
print_arguments(parser, args)

Arguments and Data Types:
  target_size: tuple_type - (240, 240)
  batch_size: int - 16
  train_frac: float - 0.8
  random_state: int - 42
  num_classes: int - 2
  channels: int - 3
  output_dir: None - output
  augmentation_types: str - ['flip', 'rotation']
  images: None - data/galaxies
  epochs: int - 2
  model_path: None - None
  val_dir: None - None


## Define AlexNet architecture

In [9]:
model = AlexNet.build(width=args.target_size[0], height=args.target_size[1], \
                      depth=args.channels, classes=1, reg=0.0002)

/data/linn/miniconda3/envs/aiml/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 120, 120, 96)   │         7,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 120, 120, 96)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 120, 120, 96)   │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 59, 59, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 59, 59, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 59, 59, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 59, 59, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 59, 59, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 29, 29, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 29, 29, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 29, 29, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 29, 29, 384)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 29, 29, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 29, 29, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 29, 29, 384)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 29, 29, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 29, 29, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 29, 29, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 29, 29, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 226,068,225 (862.38 MB)

 Trainable params: 226,049,089 (862.31 MB)

 Non-trainable params: 19,136 (74.75 KB)

None


## Define validation loss, evaluation metrics, optimizer and learning rate

In [11]:
classification_threshold = 0.5

METRICS = [
      tf.keras.metrics.Precision(thresholds=classification_threshold,
                                 name='precision'),
      tf.keras.metrics.Recall(thresholds=classification_threshold,
                              name="recall"),
      tf.keras.metrics.AUC(num_thresholds=100, curve='PR', name='auc_pr'),
]

In [12]:
model.compile(loss="binary_crossentropy", optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3)\
              , metrics=METRICS)

## Define rescale and custom augmentations

In [13]:
def random_choice(x, size, seed, axis=0, unique=True):
    dim_x = tf.cast(tf.shape(x)[axis], tf.int64)
    indices = tf.range(0, dim_x, dtype=tf.int64)
    sample_index = tf.random.shuffle(indices,seed=seed)[:size]
    sample = tf.gather(x, sample_index, axis=axis)

    return sample, sample_index

def random_int_rot_img(inputs,seed):
    angles = tf.constant([1, 2, 3, 4])
    # Make a new seed.
    #new_seed = tf.random.experimental.stateless_split((seed,seed), num=1)[0, :]
    angle = random_choice(angles,1,seed=seed)[0][0]
    inputs = tf.image.rot90(inputs, k=angle)

    return inputs

def rescale(image, label):
    image = tf.cast(image, tf.float32)
    image = (image / 255.0)

    return image, label

# define custom augmentations
def augment_custom(images, labels, augmentation_types, seed):
    images, labels = rescale(images, labels)
    # Make a new seed.
    #new_seed = tf.random.experimental.stateless_split((seed,seed), num=1)[0, :]
    new_seed = seed
    if 'rotation' in augmentation_types:
        images = random_int_rot_img(images,seed=seed)
    if 'flip' in augmentation_types:
        images = tf.image.random_flip_left_right(images, seed=new_seed)
        images = tf.image.random_flip_up_down(images, seed=new_seed)
    if 'brightness' in augmentation_types:
        images = tf.image.random_brightness(images, max_delta=0.2, seed=new_seed)
    if 'contrast' in augmentation_types:
        images = tf.image.random_contrast(images, lower=0.2, upper=0.5, seed=new_seed)

    return (images, labels)

## Define Callbacks

In [14]:
class SaveHistoryCallback(Callback):
    def __init__(self, file_path):
        super().__init__()
        self.file_path = file_path
        self.history = {'loss': [], 'val_loss': [], 'auc_pr':[], 'val_auc_pr':[], 'val_precision':[], 'val_recall':[]}

    def on_epoch_end(self, epoch, logs=None):
        self.history['loss'].append(logs.get('loss'))
        self.history['val_loss'].append(logs.get('val_loss'))
        self.history['auc_pr'].append(logs.get('auc_pr'))
        self.history['val_auc_pr'].append(logs.get('val_auc_pr'))
        self.history['val_precision'].append(logs.get('val_precision'))
        self.history['val_recall'].append(logs.get('val_recall'))

        with open(self.file_path, 'w') as f:
            json.dump(self.history, f)

In [15]:
def create_callbacks(run_name):
    outdir = os.path.join("output", run_name)
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    model_path = os.path.join(outdir,"best_model.keras")
    mc = ModelCheckpoint(model_path, monitor='val_loss', \
        mode='min', verbose=1, save_best_only=True)
    history_path = os.path.join(outdir,'history.json')
    hc = SaveHistoryCallback(history_path)
    callbacks=[mc,hc, wandb.keras.WandbMetricsLogger()]
    return callbacks

## Create tf.data.Dataset

In [16]:
def get_train_data(data_dir, val_dir, train_frac, target_size, batch_size, augmentation_types, outdir, random_state):
  if val_dir is None:
      train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=1-train_frac,
        subset="both",
        color_mode='rgb',
        seed=random_state,
        image_size=target_size,
        batch_size=None)
  else:
      train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        color_mode='rgb',
        seed=random_state,
        image_size=target_size,
        batch_size=None)

      val_ds = tf.keras.utils.image_dataset_from_directory(
            val_dir,
            color_mode='rgb',
            seed=random_state,
            image_size=target_size,
            batch_size=None)

  class_names = train_ds.class_names
  print("Training dataset class names are :",class_names)

  AUTOTUNE = tf.data.AUTOTUNE

  train_ds = (
          train_ds
          .shuffle(1000)
          .map(lambda x, y: augment_custom(x, y, augmentation_types, seed=random_state), num_parallel_calls=AUTOTUNE)
          #.cache()
          .batch(batch_size)
          .prefetch(buffer_size=AUTOTUNE)
          )

  val_ds = (
          val_ds
          .map(rescale, num_parallel_calls=AUTOTUNE)
          #.cache()
          .batch(batch_size)
          .prefetch(buffer_size=AUTOTUNE)
          )

  return train_ds, val_ds

In [17]:
train_ds, val_ds = get_train_data(args.images, args.val_dir, args.train_frac, args.target_size, args.batch_size,\
                                  args.augmentation_types, args.output_dir, args.random_state)

Found 15229 files belonging to 2 classes.
Using 12184 files for training.
Using 3045 files for validation.
Training dataset class names are : ['NonRings', 'Rings']


In [18]:
wandb.init(project="aiml-handson", anonymous="allow")
callbacks = create_callbacks(wandb.run.name)
history = model.fit(train_ds, validation_data=val_ds, epochs=args.epochs, shuffle=True, callbacks=callbacks)
wandb.finish()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: linn-official. Use `wandb login --relogin` to force relogin


Epoch 1/2
762/762 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - auc_pr: 0.0868 - loss: 3.7906 - precision: 0.0909 - recall: 0.1485   
Epoch 1: val_loss improved from inf to 2.08330, saving model to output/eager-hill-4/best_model.keras


NameError: name 'json' is not defined